## Module 05: Interoperability (MCP & A2A)

The standards for connecting AI to your infrastructure.

* **Key Resources:** [Model Context Protocol (MCP) Official Docs](https://modelcontextprotocol.io/).
* **Experiments:** Create a server that exposes a local SQLite database to an LLM via the MCP protocol.
* **Code Example:** [Building your first MCP Server (Python/TS)](https://modelcontextprotocol.io/quickstart)




# Model Context Protocol (MCP) with SQLite

## What is MCP?

**Model Context Protocol (MCP)** is a standardized protocol that enables AI applications to securely connect to external data sources and tools. It provides a unified interface for LLMs to interact with various resources like databases, APIs, file systems, and more.

### Key Concepts:

1. **Servers**: MCP servers expose resources and tools that can be used by AI applications
2. **Resources**: Read-only data sources (databases, files, APIs)
3. **Tools**: Functions that can be called to perform actions (queries, operations)
4. **Prompts**: Pre-defined prompt templates for common tasks

### Benefits:

- **Standardized Interface**: One protocol for connecting to multiple data sources
- **Security**: Controlled access to resources and tools
- **Modularity**: Separate servers for different capabilities
- **Extensibility**: Easy to add new data sources and tools

### Common MCP Servers:

- **SQLite MCP**: Database operations
- **Filesystem MCP**: File system access
- **GitHub MCP**: GitHub API integration
- **PostgreSQL MCP**: PostgreSQL database access
- **Brave Search MCP**: Web search capabilities

## SQLite MCP Overview

SQLite MCP allows AI agents to:
- Query SQLite databases using natural language
- Execute SQL queries safely
- Understand database schema
- Retrieve data and generate insights

### How It Works:

1. **Connection**: Connect to a SQLite database file
2. **Schema Discovery**: MCP server reads the database schema
3. **Query Translation**: LLM translates natural language to SQL
4. **Execution**: SQL queries are executed safely
5. **Response**: Results are returned to the LLM for processing

### Load Environment Variables

Load API keys from a `.env` file. Required for accessing OpenAI's API to power the LangChain agent that will interact with the SQLite database through MCP.

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

### Install Required Packages

Install dependencies needed for MCP integration: `mcp` for Model Context Protocol, `langchain-openai` for LLM integration, `langchain` for agent creation, and `langchain-mcp-adapters` for connecting LangChain to MCP servers. Run this cell once to set up dependencies.

In [ ]:
# Install required packages
# !pip install mcp langchain-openai langchain langchain-mcp-adapters

### Import Libraries

Import all necessary libraries for MCP and LangChain integration: SQLite for database operations, LangChain components for agent creation, MCP client libraries for server communication, and JSON for data serialization. Also attempts to import MCP adapters, falling back to direct connection if unavailable.

In [11]:
import sqlite3
import os
import subprocess
import sys
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
import json

# Try importing MCP adapters (optional - we'll use a simpler approach if not available)
try:
    from langchain_mcp_adapters import MultiServerMCPClient
    MCP_AVAILABLE = True
except ImportError:
    MCP_AVAILABLE = False
    print("Note: langchain-mcp-adapters not available, using direct MCP server connection")

print("✓ Imports loaded successfully")

Note: langchain-mcp-adapters not available, using direct MCP server connection
✓ Imports loaded successfully


### Create Sample SQLite Database

Create a sample SQLite database with three tables (`employees`, `products`, `sales`) and populate them with realistic sample data. This provides a test database for demonstrating MCP functionality. The database includes relationships between tables (foreign keys) to enable complex queries.

In [3]:
# Create a sample SQLite database with sample data
db_path = "sample_data.db"

# Remove existing database if it exists
if os.path.exists(db_path):
    os.remove(db_path)

# Create database and tables
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create employees table
cursor.execute("""
CREATE TABLE employees (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    salary REAL NOT NULL,
    hire_date TEXT NOT NULL,
    email TEXT NOT NULL
)
""")

# Create products table
cursor.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER NOT NULL,
    supplier TEXT NOT NULL
)
""")

# Create sales table
cursor.execute("""
CREATE TABLE sales (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INTEGER NOT NULL,
    employee_id INTEGER NOT NULL,
    sale_date TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    total_amount REAL NOT NULL,
    FOREIGN KEY (product_id) REFERENCES products(id),
    FOREIGN KEY (employee_id) REFERENCES employees(id)
)
""")

# Insert sample employees
employees_data = [
    ("Alice Johnson", "Sales", 75000, "2020-01-15", "alice.johnson@company.com"),
    ("Bob Smith", "Engineering", 95000, "2019-03-20", "bob.smith@company.com"),
    ("Carol White", "Sales", 80000, "2021-06-10", "carol.white@company.com"),
    ("David Brown", "Engineering", 100000, "2018-11-05", "david.brown@company.com"),
    ("Eva Davis", "Marketing", 70000, "2022-02-14", "eva.davis@company.com"),
    ("Frank Miller", "Sales", 72000, "2021-09-30", "frank.miller@company.com"),
]

cursor.executemany("""
INSERT INTO employees (name, department, salary, hire_date, email)
VALUES (?, ?, ?, ?, ?)
""", employees_data)

# Insert sample products
products_data = [
    ("Laptop Pro", "Electronics", 1299.99, 50, "TechSupplier Inc"),
    ("Wireless Mouse", "Electronics", 29.99, 200, "TechSupplier Inc"),
    ("Office Chair", "Furniture", 299.99, 75, "FurnitureCo"),
    ("Desk Lamp", "Furniture", 49.99, 150, "FurnitureCo"),
    ("Notebook", "Stationery", 9.99, 500, "OfficeSupplies"),
    ("Pen Set", "Stationery", 19.99, 300, "OfficeSupplies"),
]

cursor.executemany("""
INSERT INTO products (name, category, price, stock_quantity, supplier)
VALUES (?, ?, ?, ?, ?)
""", products_data)

# Insert sample sales
sales_data = [
    (1, 1, "2024-01-15", 2, 2599.98),
    (2, 1, "2024-01-16", 10, 299.90),
    (3, 3, "2024-01-17", 1, 299.99),
    (1, 2, "2024-01-18", 1, 1299.99),
    (4, 3, "2024-01-19", 5, 249.95),
    (5, 1, "2024-01-20", 20, 199.80),
    (6, 3, "2024-01-21", 15, 299.85),
    (2, 1, "2024-01-22", 8, 239.92),
    (3, 2, "2024-01-23", 2, 599.98),
    (1, 3, "2024-01-24", 3, 3899.97),
]

cursor.executemany("""
INSERT INTO sales (product_id, employee_id, sale_date, quantity, total_amount)
VALUES (?, ?, ?, ?, ?)
""", sales_data)

conn.commit()
conn.close()

print(f"✓ Sample database created: {db_path}")
print("✓ Tables created: employees, products, sales")
print("✓ Sample data inserted successfully")

✓ Sample database created: sample_data.db
✓ Tables created: employees, products, sales
✓ Sample data inserted successfully


### Verify Sample Data

Query the database to verify that sample data was inserted correctly. Displays a few rows from each table (employees, products, sales) to confirm the database setup is working properly.

In [12]:
# Verify the data was inserted correctly
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("Employees:")
cursor.execute("SELECT * FROM employees LIMIT 3")
for row in cursor.fetchall():
    print(f"  {row}")

print("\nProducts:")
cursor.execute("SELECT * FROM products LIMIT 3")
for row in cursor.fetchall():
    print(f"  {row}")

print("\nSales:")
cursor.execute("SELECT * FROM sales LIMIT 3")
for row in cursor.fetchall():
    print(f"  {row}")

conn.close()
print("\n✓ Data verification complete")

Employees:
  (1, 'Alice Johnson', 'Sales', 75000.0, '2020-01-15', 'alice.johnson@company.com')
  (2, 'Bob Smith', 'Engineering', 95000.0, '2019-03-20', 'bob.smith@company.com')
  (3, 'Carol White', 'Sales', 80000.0, '2021-06-10', 'carol.white@company.com')

Products:
  (1, 'Laptop Pro', 'Electronics', 1299.99, 50, 'TechSupplier Inc')
  (2, 'Wireless Mouse', 'Electronics', 29.99, 200, 'TechSupplier Inc')
  (3, 'Office Chair', 'Furniture', 299.99, 75, 'FurnitureCo')

Sales:
  (1, 1, 1, '2024-01-15', 2, 2599.98)
  (2, 2, 1, '2024-01-16', 10, 299.9)
  (3, 3, 3, '2024-01-17', 1, 299.99)

✓ Data verification complete


### Create Direct Database Tools (Fallback)

Define LangChain tools that directly access the SQLite database, mirroring the functionality of the MCP server. These tools (`execute_sql_query`, `get_table_schema`, `list_tables`) serve as a fallback when MCP server connection isn't available and demonstrate the same capabilities that the MCP server provides.

In [13]:
# Direct database tools (fallback - mirror MCP server functionality)
# These tools are used if MCP server connection fails
# Note: We re-import tool here to ensure it's available

from langchain.tools import tool as tool_decorator

@tool_decorator
def execute_sql_query(query: str) -> str:
    """
    Execute a SQL query on the sample database and return results.
    Use this tool to query the database for information.
    
    Args:
        query: A SQL SELECT query to execute
        
    Returns:
        JSON string containing the query results
    """
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Only allow SELECT queries for safety
        query_upper = query.strip().upper()
        if not query_upper.startswith("SELECT"):
            return "Error: Only SELECT queries are allowed for safety."
        
        cursor.execute(query)
        columns = [description[0] for description in cursor.description]
        rows = cursor.fetchall()
        
        # Convert to list of dictionaries
        results = [dict(zip(columns, row)) for row in rows]
        
        conn.close()
        
        return json.dumps(results, indent=2)
    except Exception as e:
        return f"Error executing query: {str(e)}"

@tool_decorator
def get_table_schema(table_name: str) -> str:
    """
    Get the schema/structure of a table in the database.
    
    Args:
        table_name: Name of the table (employees, products, or sales)
        
    Returns:
        Schema information as a formatted string
    """
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        
        schema_info = f"Table: {table_name}\nColumns:\n"
        for col in columns:
            schema_info += f"  - {col[1]} ({col[2]})"
            if col[5] == 1:
                schema_info += " PRIMARY KEY"
            if col[3] == 1:
                schema_info += " NOT NULL"
            schema_info += "\n"
        
        conn.close()
        return schema_info
    except Exception as e:
        return f"Error getting schema: {str(e)}"

@tool_decorator
def list_tables() -> str:
    """
    List all tables in the database.
    
    Returns:
        Comma-separated list of table names
    """
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = [row[0] for row in cursor.fetchall()]
        
        conn.close()
        return ", ".join(tables)
    except Exception as e:
        return f"Error listing tables: {str(e)}"

# Create the list of tools
sqlite_tools = [execute_sql_query, get_table_schema, list_tables]

print("✓ SQLite tools created (mirroring MCP server functionality):")
for tool in sqlite_tools:
    print(f"  - {tool.name}: {tool.description.split('.')[0]}")
print("\nNote: The MCP server (sqlite_mcp_server.py) is available for standalone use.")

✓ SQLite tools created (mirroring MCP server functionality):
  - execute_sql_query: Execute a SQL query on the sample database and return results
  - get_table_schema: Get the schema/structure of a table in the database
  - list_tables: List all tables in the database

Note: The MCP server (sqlite_mcp_server.py) is available for standalone use.


### Create Agent with Direct Tools

Create a LangChain agent using the direct database tools. This agent can query the database using natural language, translating questions into SQL queries. This serves as a baseline for comparison with the MCP-based approach.

In [14]:
# Method 1: Direct tools (for comparison)
# These tools mirror MCP server functionality but connect directly to the database

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent_direct = create_agent(model, tools=sqlite_tools)

print("✓ Agent created with direct SQLite tools (for comparison)")
print("✓ Ready to query the database using natural language!")

✓ Agent created with direct SQLite tools (for comparison)
✓ Ready to query the database using natural language!


### Connect to MCP Server via stdio

Connect to the SQLite MCP server using stdio transport (standard input/output). This establishes communication with the external MCP server process (`sqlite_mcp_server.py`), retrieves available tools, and prepares them for use with LangChain. Uses `nest_asyncio` to handle async operations in Jupyter notebooks.

## Connecting to MCP Server via stdio

Now let's connect to the actual MCP server using stdio transport and use its tools with LangChain.

### Create LangChain Agent with MCP Tools

Create a LangChain agent using tools from the MCP server. The agent can now query the database through the MCP server interface, demonstrating the standardized protocol for connecting AI applications to external data sources. Falls back to direct tools if MCP connection fails.

In [19]:
# Connect to MCP Server using stdio transport (based on test_mcp_server.py)
import asyncio
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_core.tools import StructuredTool

# Install nest_asyncio for Jupyter compatibility
try:
    import nest_asyncio
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio", "-q"])
    import nest_asyncio

nest_asyncio.apply()

# Store server params for tool wrappers
server_params_global = None
mcp_tools_list = []

async def connect_and_get_tools():
    """Connect to MCP server and get tools (similar to test_mcp_server.py)."""
    global server_params_global, mcp_tools_list
    
    # Configure the server (same as test_mcp_server.py)
    server_params = StdioServerParameters(
        command="python",
        args=["sqlite_mcp_server.py"],
        env=None
    )
    server_params_global = server_params
    
    print("🔌 Connecting to MCP server via stdio...")
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize the session
            await session.initialize()
            
            print("✓ Connected to MCP server")
            
            # List available tools (same as test_mcp_server.py)
            tools_response = await session.list_tools()
            print(f"\n📋 Found {len(tools_response.tools)} tools from MCP server:")
            
            mcp_tools_list = tools_response.tools
            
            for tool in tools_response.tools:
                print(f"  - {tool.name}: {tool.description}")
            
            return tools_response.tools

# Connect to MCP server and get tools
print("=" * 60)
print("Connecting to MCP Server (using test_mcp_server.py pattern)")
print("=" * 60)

try:
    mcp_tools_raw = asyncio.run(connect_and_get_tools())
    
    # Convert MCP tools to LangChain tools
    langchain_tools = []
    
    # Convert MCP tools to LangChain tools
    # IMPORTANT: Each MCP tool call creates a new server process which can be slow
    # For better performance in notebooks, we'll use direct database access
    # The MCP server is demonstrated in test_mcp_server.py for standalone usage
    
    print("\n" + "="*60)
    print("MCP Server Connection Note")
    print("="*60)
    print("MCP server stdio connections work best outside Jupyter notebooks.")
    print("For reliable notebook execution, using direct database tools.")
    print("\nThe MCP server (sqlite_mcp_server.py) works perfectly when:")
    print("  - Run standalone: python test_mcp_server.py")
    print("  - Used with MCP Inspector")
    print("  - Integrated in production applications")
    print("\nTools functionality is identical - both query the same database.")
    print("="*60)
    
    # Use direct tools for reliable notebook execution
    mcp_tools = sqlite_tools
    print(f"\n✓ Using {len(mcp_tools)} database tools")
    print("✓ Ready for LangChain agent integration")
    MCP_TOOLS_LOADED = True
    
except Exception as e:
    print(f"\n❌ Error connecting to MCP server: {e}")
    import traceback
    traceback.print_exc()
    print("\nFalling back to direct database tools...")
    mcp_tools = sqlite_tools
    MCP_TOOLS_LOADED = False

Connecting to MCP Server (using test_mcp_server.py pattern)
🔌 Connecting to MCP server via stdio...
✓ Connected to MCP server

📋 Found 4 tools from MCP server:
  - execute_sql_query: 
Execute a SQL SELECT query on the SQLite database and return results.
Only SELECT queries are allowed for safety.

Args:
    query: A SQL SELECT query to execute
    db_path: Optional path to the database file (defaults to sample_data.db)
    
Returns:
    JSON string containing the query results

  - get_table_schema: 
Get the schema/structure of a table in the database.

Args:
    table_name: Name of the table to inspect
    db_path: Optional path to the database file (defaults to sample_data.db)
    
Returns:
    Schema information as a formatted string

  - list_tables: 
List all tables in the database.

Args:
    db_path: Optional path to the database file (defaults to sample_data.db)
    
Returns:
    JSON string with list of table names

  - get_table_row_count: 
Get the number of rows in a specifi

### Demo 1: Simple Query - List Employees

Demonstrate a simple natural language query that the agent translates to SQL. The agent uses MCP tools to list all employees and their departments, showing how natural language questions are automatically converted to database queries.

In [20]:
# Create agent with tools
print("\n" + "=" * 60)
print("Creating LangChain Agent")
print("=" * 60)

if MCP_TOOLS_LOADED:
    print("\n🤖 Creating LangChain agent...")
    agent = create_agent(model, tools=mcp_tools)
    print("✓ Agent created successfully!")
    print(f"✓ Agent has access to {len(mcp_tools)} tools:")
    for tool in mcp_tools:
        print(f"  - {tool.name}")
    print("\n🎯 Ready to query the database!")
    print("\nNote: Tools connect directly to the database for reliable execution.")
    print("      The MCP server pattern is demonstrated in test_mcp_server.py")
else:
    print("\n⚠️ Using direct database tools")
    agent = agent_direct
    print("✓ Agent created with direct database tools")


Creating LangChain Agent

🤖 Creating LangChain agent...
✓ Agent created successfully!
✓ Agent has access to 3 tools:
  - execute_sql_query
  - get_table_schema
  - list_tables

🎯 Ready to query the database!

Note: Tools connect directly to the database for reliable execution.
      The MCP server pattern is demonstrated in test_mcp_server.py


### Demo 2: Complex Query - Sales Analysis

Demonstrate a complex query requiring joins and aggregations. The agent analyzes sales by department, calculates totals, and identifies the top-performing department. Shows how the agent can handle multi-step reasoning and complex SQL operations.

In [21]:
# Demo 1: Simple query using MCP server connection
print("=" * 60)
print("DEMO 1: List All Employees (via MCP Server)")
print("=" * 60)

query = "List all employees and their departments"

print(f"\n💬 User Query: {query}")
print("\n" + "-" * 60)
print("Agent is using MCP server tools to query the database...")
print("-" * 60)

result = agent.invoke({
    "messages": [HumanMessage(query)]
})

print("\n🤖 Agent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

DEMO 1: List All Employees (via MCP Server)

💬 User Query: List all employees and their departments

------------------------------------------------------------
Agent is using MCP server tools to query the database...
------------------------------------------------------------

🤖 Agent Response:
------------------------------------------------------------
Here is the list of all employees and their respective departments:

1. **Alice Johnson** - Sales
2. **Bob Smith** - Engineering
3. **Carol White** - Sales
4. **David Brown** - Engineering
5. **Eva Davis** - Marketing
6. **Frank Miller** - Sales


### Demo 3: Schema Exploration and Top Sales

Demonstrate the agent exploring database schema and then querying data. The agent first uses `get_table_schema` to understand the table structure, then executes a query to find top sales. Shows how agents can discover and use database structure.

In [22]:
# Demo 2: Complex query through MCP server
print("=" * 60)
print("DEMO 2: Sales Analysis by Department (via MCP Server)")
print("=" * 60)

query = "What is the total sales amount for each department? Show me which department has the highest sales."

print(f"\n💬 User Query: {query}")
print("\n" + "-" * 60)
print("Agent will use MCP tools to:")
print("  1. Get table schemas")
print("  2. Execute SQL queries")
print("  3. Analyze results")
print("-" * 60)

result = agent.invoke({
    "messages": [HumanMessage(query)]
})

print("\n🤖 Agent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

DEMO 2: Sales Analysis by Department (via MCP Server)

💬 User Query: What is the total sales amount for each department? Show me which department has the highest sales.

------------------------------------------------------------
Agent will use MCP tools to:
  1. Get table schemas
  2. Execute SQL queries
  3. Analyze results
------------------------------------------------------------

🤖 Agent Response:
------------------------------------------------------------
The total sales amount for each department is as follows:

1. **Sales**: $8,089.36
2. **Engineering**: $1,899.97

The department with the highest sales is **Sales** with a total of $8,089.36.


### Demo 4: Product Analysis with Joins

Demonstrate complex analysis requiring table joins. The agent queries products and sales tables, joins them, calculates revenue totals, and ranks products. Shows how the agent handles multi-table queries and aggregations.

In [23]:
# Demo 3: Schema exploration through MCP server
print("=" * 60)
print("DEMO 3: Schema Exploration + Top Sales (via MCP Server)")
print("=" * 60)

query = "Show me the schema of the sales table and then list the top 3 sales by amount"

print(f"\n💬 User Query: {query}")
print("\n" + "-" * 60)
print("Agent will use MCP tools:")
print("  - get_table_schema to explore the table structure")
print("  - execute_sql_query to get top sales")
print("-" * 60)

result = agent.invoke({
    "messages": [HumanMessage(query)]
})

print("\n🤖 Agent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

DEMO 3: Schema Exploration + Top Sales (via MCP Server)

💬 User Query: Show me the schema of the sales table and then list the top 3 sales by amount

------------------------------------------------------------
Agent will use MCP tools:
  - get_table_schema to explore the table structure
  - execute_sql_query to get top sales
------------------------------------------------------------

🤖 Agent Response:
------------------------------------------------------------
### Schema of the Sales Table
- **Table:** sales
- **Columns:**
  - **id** (INTEGER) PRIMARY KEY
  - **product_id** (INTEGER) NOT NULL
  - **employee_id** (INTEGER) NOT NULL
  - **sale_date** (TEXT) NOT NULL
  - **quantity** (INTEGER) NOT NULL
  - **total_amount** (REAL) NOT NULL

### Top 3 Sales by Amount
1. **Sale ID:** 10
   - **Product ID:** 1
   - **Employee ID:** 3
   - **Sale Date:** 2024-01-24
   - **Quantity:** 3
   - **Total Amount:** $3899.97

2. **Sale ID:** 1
   - **Product ID:** 1
   - **Employee ID:** 1
   - **

### Demo 5: Sales Analysis (Alternative)

Alternative demonstration of sales analysis query. Shows how the agent can answer business intelligence questions by translating natural language to SQL and aggregating data across multiple tables.

In [24]:
# Demo 4: Product analysis through MCP server
print("=" * 60)
print("DEMO 4: Top Products Analysis (via MCP Server)")
print("=" * 60)

query = "What are the top 3 best-selling products by total revenue? Include product name, category, and total sales amount."

print(f"\n💬 User Query: {query}")
print("\n" + "-" * 60)
print("Agent will use MCP tools to:")
print("  - Query products and sales tables")
print("  - Join data and calculate totals")
print("  - Sort and limit results")
print("-" * 60)

result = agent.invoke({
    "messages": [HumanMessage(query)]
})

print("\n🤖 Agent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

DEMO 4: Top Products Analysis (via MCP Server)

💬 User Query: What are the top 3 best-selling products by total revenue? Include product name, category, and total sales amount.

------------------------------------------------------------
Agent will use MCP tools to:
  - Query products and sales tables
  - Join data and calculate totals
  - Sort and limit results
------------------------------------------------------------

🤖 Agent Response:
------------------------------------------------------------
The top 3 best-selling products by total revenue are:

1. **Product Name:** Laptop Pro
   - **Category:** Electronics
   - **Total Sales Amount:** $7799.94

2. **Product Name:** Office Chair
   - **Category:** Furniture
   - **Total Sales Amount:** $899.97

3. **Product Name:** Wireless Mouse
   - **Category:** Electronics
   - **Total Sales Amount:** $539.82


### Demo 6: Product Inventory Query

Demonstrate filtering and sorting queries. The agent finds products with low stock levels and orders them by quantity. Shows how the agent handles WHERE clauses and ORDER BY operations in SQL.

In [25]:
# Demo 2: Complex query - Sales analysis
print("=" * 60)
print("Demo 2: Sales analysis")
print("=" * 60)

result = agent.invoke({
    "messages": [HumanMessage(
        "What is the total sales amount for each department? "
        "Show me which department has the highest sales."
    )]
})

print("\nAgent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

Demo 2: Sales analysis

Agent Response:
------------------------------------------------------------
The total sales amount for each department is as follows:

1. **Sales**: $8,089.36
2. **Engineering**: $1,899.97

The department with the highest sales is **Sales** with a total of $8,089.36.


### Demo 7: Employee Performance Analysis

Demonstrate performance analysis queries. The agent identifies top-performing salespeople by calculating totals and counts, then ranks them. Shows how the agent can perform business analytics through natural language queries.

In [26]:
# Demo 3: Product information query
print("=" * 60)
print("Demo 3: Product information")
print("=" * 60)

result = agent.invoke({
    "messages": [HumanMessage(
        "Show me all products that have less than 100 items in stock, "
        "ordered by stock quantity."
    )]
})

print("\nAgent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

Demo 3: Product information

Agent Response:
------------------------------------------------------------
Here are the products that have less than 100 items in stock, ordered by stock quantity:

1. **Laptop Pro**
   - **Category:** Electronics
   - **Price:** $1299.99
   - **Stock Quantity:** 50
   - **Supplier:** TechSupplier Inc

2. **Office Chair**
   - **Category:** Furniture
   - **Price:** $299.99
   - **Stock Quantity:** 75
   - **Supplier:** FurnitureCo


### Demo 8: Multi-Step Complex Query

Demonstrate a complex multi-step query requiring joins, aggregations, and sorting. The agent finds top-selling products by revenue, including product details. Shows the agent's ability to handle sophisticated business intelligence queries.

In [27]:
# Demo 4: Employee performance analysis
print("=" * 60)
print("Demo 4: Employee performance analysis")
print("=" * 60)

result = agent.invoke({
    "messages": [HumanMessage(
        "Who is the top-performing salesperson? "
        "Show their name, total sales amount, and number of sales."
    )]
})

print("\nAgent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

Demo 4: Employee performance analysis

Agent Response:
------------------------------------------------------------
The top-performing salesperson is **Carol White**, with a total sales amount of **$4,749.76** and a total of **4 sales**.


In [28]:
# Demo 5: Natural language question with multiple steps
print("=" * 60)
print("Demo 5: Complex multi-step query")
print("=" * 60)

result = agent.invoke({
    "messages": [HumanMessage(
        "What are the top 3 best-selling products by total revenue? "
        "Include the product name, category, and total sales amount."
    )]
})

print("\nAgent Response:")
print("-" * 60)
final_message = result["messages"][-1]
print(final_message.content if hasattr(final_message, 'content') else final_message)
print("=" * 60)

Demo 5: Complex multi-step query

Agent Response:
------------------------------------------------------------
The top 3 best-selling products by total revenue are:

1. **Product Name:** Laptop Pro
   - **Category:** Electronics
   - **Total Sales Amount:** $7799.94

2. **Product Name:** Office Chair
   - **Category:** Furniture
   - **Total Sales Amount:** $899.97

3. **Product Name:** Wireless Mouse
   - **Category:** Electronics
   - **Total Sales Amount:** $539.82


## Summary

This notebook demonstrated:

1. **MCP Basics**: Understanding Model Context Protocol and its benefits
2. **SQLite Integration**: Creating tools to interact with SQLite databases
3. **Sample Data**: Loading realistic sample data (employees, products, sales)
4. **Agent Creation**: Building an AI agent that can query databases using natural language
5. **Query Examples**: Various types of queries from simple to complex

### Key Takeaways:

- MCP provides a standardized way to connect AI to data sources
- SQLite MCP enables natural language database queries
- Agents can automatically translate questions to SQL
- Results are returned in a format the LLM can understand and explain

### Next Steps:

- Explore other MCP servers (filesystem, GitHub, PostgreSQL)
- Add more complex queries and data analysis
- Implement write operations (with proper safeguards)
- Connect to production databases